# Calendar Dimension Table

Builds the standard calendar table `adb_newcaresync.silver.dim_calendar` with one row per day from 1940-01-01 to 2050-12-31, including date keys, year/quarter/month/week/day attributes, period boundaries and weekend/leap-year flags.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_newcaresync.silver.dim_calendar
COMMENT 'Standard calendar dimension: one row per day from 1940-01-01 to 2050-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2050-12-31', INTERVAL 1 DAY)) AS d
),
base AS (
  SELECT d, ((dayofweek(d) + 5) % 7) + 1 AS iso_dow  -- 1 = Monday ... 7 = Sunday
  FROM dates
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)              AS date_key,
  d                                                    AS full_date,
  -- Year
  year(d)                                              AS date_year,
  CASE WHEN (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 THEN TRUE ELSE FALSE END AS is_leap_year,
  -- Half / Quarter
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END            AS date_half_year,
  quarter(d)                                           AS date_quarter,
  concat('Q', quarter(d))                              AS quarter_name,
  concat(year(d), '-Q', quarter(d))                    AS year_quarter,
  -- Month
  month(d)                                             AS date_month,
  date_format(d, 'MMMM')                               AS month_name,
  date_format(d, 'MMM')                                AS month_short_name,
  date_format(d, 'yyyy-MM')                            AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                AS year_month_key,
  day(last_day(d))                                     AS days_in_month,
  -- Week (ISO 8601)
  weekofyear(d)                                        AS iso_week_of_year,
  year(date_add(d, 4 - iso_dow))                       AS iso_year,
  date_sub(d, iso_dow - 1)                             AS week_start_date,
  date_add(d, 7 - iso_dow)                             AS week_end_date,
  -- Day
  day(d)                                               AS day_of_month,
  iso_dow                                              AS day_of_week,
  date_format(d, 'EEEE')                               AS day_name,
  date_format(d, 'EEE')                                AS day_short_name,
  dayofyear(d)                                         AS day_of_year,
  CAST(ceil(day(d) / 7.0) AS INT)                      AS week_of_month,
  -- Period boundaries
  trunc(d, 'MM')                                       AS first_day_of_month,
  last_day(d)                                          AS last_day_of_month,
  trunc(d, 'QUARTER')                                  AS first_day_of_quarter,
  date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)      AS last_day_of_quarter,
  trunc(d, 'YEAR')                                     AS first_day_of_year,
  make_date(year(d), 12, 31)                           AS last_day_of_year,
  -- Flags
  iso_dow IN (6, 7)                                    AS is_weekend,
  iso_dow NOT IN (6, 7)                                AS is_weekday,
  d = trunc(d, 'MM')                                   AS is_month_start,
  d = last_day(d)                                      AS is_month_end,
  d = trunc(d, 'QUARTER')                              AS is_quarter_start,
  d = date_sub(add_months(trunc(d, 'QUARTER'), 3), 1)  AS is_quarter_end,
  d = trunc(d, 'YEAR')                                 AS is_year_start,
  d = make_date(year(d), 12, 31)                       AS is_year_end
FROM base
ORDER BY d;

In [0]:
%sql
SELECT
  count(*)                     AS total_rows,
  count(DISTINCT date_key)     AS distinct_keys,
  min(full_date)               AS min_date,
  max(full_date)               AS max_date,
  datediff(max(full_date), min(full_date)) + 1 AS expected_rows
FROM adb_newcaresync.silver.dim_calendar;

In [0]:
%sql
SELECT * FROM adb_newcaresync.silver.dim_calendar
WHERE full_date BETWEEN DATE'2024-12-28' AND DATE'2025-01-05'
ORDER BY full_date;